## Импорты и настройка окружения

In [1]:
import os
import re
from pathlib import Path
from typing import List, Dict, Any

# Инструменты RAG
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import Distance, VectorParams

# Визуализация
import pandas as pd
from IPython.display import display, Markdown

# Конфиг
DATA_DIR = Path("../data/knowledge_base")
COLLECTION_NAME = "robotex_docs"
QDRANT_URL = "http://localhost:6333"
EMBEDDING_MODEL = "BAAI/bge-m3"

# Проверка GPU
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Device: {device.upper()}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

🚀 Device: CUDA
   GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## Загрузка и "хитрый" парсинг

In [2]:
def load_and_enrich_documents(directory: str) -> List[Any]:
    """
    Загружает MD файлы и извлекает метаданные из текста (эмуляция парсинга заголовков).
    """
    loader = DirectoryLoader(directory, glob="*.md", loader_cls=TextLoader)
    raw_docs = loader.load()
    
    enriched_docs = []
    for doc in raw_docs:
        content = doc.page_content
        meta = doc.metadata
        
        # 1. Извлекаем Год (ищем "2024", "2025", "2026")
        # Если года нет явно, считаем 2026 (текущий)
        years = re.findall(r"202[4-6]", content)
        meta["year"] = int(years[0]) if years else 2026
        
        # 2. Определение категории по имени файла
        filename = Path(meta["source"]).name
        if "HR" in filename or "Holiday" in filename:
            meta["category"] = "HR"
        elif "Security" in filename or "VPN" in filename:
            meta["category"] = "IT_Security"
        elif "Project" in filename:
            meta["category"] = "Projects"
        else:
            meta["category"] = "General"
            
        enriched_docs.append(doc)
        
    print(f"✅ Загружено {len(enriched_docs)} документов.")
    return enriched_docs

# Запускаем
docs = load_and_enrich_documents(str(DATA_DIR))

# Посмотрим на пример данных
df = pd.DataFrame([d.metadata for d in docs])
display(df.head())

✅ Загружено 30 документов.


,source,year,category
0,../data/knowledge_base/Doc_Filler_20.md,2026,General
1,../data/knowledge_base/Doc_Filler_13.md,2026,General
2,../data/knowledge_base/Doc_Filler_21.md,2026,General
3,../data/knowledge_base/Doc_Filler_24.md,2026,General
4,../data/knowledge_base/Doc_Filler_5.md,2026,General


## Чанкинг (Стратегия Recursive)

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Достаточно крупно, чтобы сохранить смысл абзаца
    chunk_overlap=200,    # Нахлест, чтобы не терять контекст на границах
    separators=["\n## ", "\n\n", "\n", " ", ""] # Приоритет разделителей (заголовки -> абзацы)
)

splits = text_splitter.split_documents(docs)
print(f"📚 Исходных документов: {len(docs)}")
print(f"✂️  Получено чанков: {len(splits)}")

# Пример чанка
print("\n--- Пример чанка ---")
print(splits[0].page_content[:200] + "...")
print("Metadata:", splits[0].metadata)

📚 Исходных документов: 30
✂️  Получено чанков: 65

--- Пример чанка ---
# Онбординг нового персонала в РобоТехе

## Цель

Целью данного документа является описание процесса онбординга для новых сотрудников в компании РобоТех. Представленная инструкция предназначена для пр...
Metadata: {'source': '../data/knowledge_base/Doc_Filler_20.md', 'year': 2026, 'category': 'General'}


## Инициализация Эмбеддингов (Local GPU)

In [4]:
print(f"⏳ Загрузка модели эмбеддингов {EMBEDDING_MODEL} на {device}...")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True} # Важно для косинусного расстояния!
)

# Тест векторизации
test_vec = embeddings.embed_query("Тестовый запрос")
print(f"✅ Вектор создан. Размерность: {len(test_vec)}") # Должно быть 1024 для bge-m3

⏳ Загрузка модели эмбеддингов BAAI/bge-m3 на cuda...


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

✅ Вектор создан. Размерность: 1024


## Создание коллекции в Qdrant (Low-Level Control)

In [ ]:
client = QdrantClient(url=QDRANT_URL)

# Проверяем, есть ли коллекция, и пересоздаем её для чистоты эксперимента
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
    print(f"🗑️ Удалена старая коллекция {COLLECTION_NAME}")

print(f"🛠 Создание коллекции с HNSW индексом...")
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
    # Настройка HNSW 
    hnsw_config=models.HnswConfigDiff(
        m=16,               # Количество связей на узел (больше = точнее, но больше памяти)
        ef_construct=100    # Глубина поиска при построении индекса
    )
)
print("✅ Коллекция готова!")

🗑️ Удалена старая коллекция robotex_docs
🛠 Создание коллекции с HNSW индексом...
✅ Коллекция готова!


## Загрузка векторов (Indexing)

In [6]:
print("⏳ Индексация документов (может занять время)...")

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

vector_store.add_documents(splits)

print(f"🎉 Успешно проиндексировано {len(splits)} чанков.")

⏳ Индексация документов (может занять время)...
🎉 Успешно проиндексировано 65 чанков.
